# COVID-19 Proteomics Data Analysis

This notebook analyzes mass spectrometry-based proteomics data from COVID-19 patient serum samples (dataset MSV000085507).

The study compares protein expression across four patient groups:
- **Severe COVID-19** – hospitalized patients with severe disease
- **Non-severe COVID-19** – patients with mild/moderate disease
- **Symptomatic non-COVID-19** – symptomatic patients who tested negative for COVID-19
- **Healthy** – healthy control subjects

## Step 1: Download the Data

Two files are retrieved from the MassIVE FTP server:
- `metadata.csv` – sample/run metadata with patient condition labels
- `quant_data.tsv` – quantitative protein abundance values (isobaric tag, TMT-based)

In [ ]:
import urllib.request

url = "ftp://anonymous@massive-ftp.ucsd.edu/x01/RMSV000000331/2021-04-28_ccms_fd9680ba/metadata/MSV000085507_covid19_patient_serum_msstats_formatted_na_to_empty_and_patient_IDs_where_available.csv"

urllib.request.urlretrieve(url, "metadata.csv")
print("Done!")

In [ ]:
import urllib.request

url = "ftp://anonymous@massive-ftp.ucsd.edu/x01/RMSV000000331/2021-04-28_ccms_fd9680ba/quant/msstats_input_isobaric_tag/58ed24a150244a91893a65c9c9333290.tsv"

print("Starting download... this may take a few minutes")
urllib.request.urlretrieve(url, "quant_data.tsv")
print("Done!")

## Step 2: Inspect Dataset Size

Before diving into analysis, we check the size of both downloaded files on disk and load the metadata to understand the structure of the experiment.

In [15]:
import os

for fname in ["metadata.csv", "quant_data.tsv"]:
    size_mb = os.path.getsize(fname) / (1024 ** 2)
    print(f"{fname}: {size_mb:.1f} MB")

metadata.csv: 0.7 MB
quant_data.tsv: 421.6 MB


In [16]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 3: Explore the Metadata

The metadata file describes each MS run: which patient it came from, their disease condition, the TMT mixture/batch, and sample channel assignments. Each row corresponds to one channel within one run.

In [17]:
import pandas as pd

df = pd.read_csv("metadata.csv")

# Overall dimensions: rows = (run × channel) combinations, cols = metadata fields
print(f"Metadata shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  → {df['Run'].nunique()} unique MS runs, {df['Subject'].nunique()} unique subjects\n")

print("Column names:")
for col in df.columns:
    print(f"  {col}")

print("\nFirst 5 rows:")
print(df.head())

Metadata shape: 5120 rows × 18 columns
  → 320 unique MS runs, 92 unique subjects

Column names:
  Dataset
  Run
  Cohort
  Subject
  Condition
  TimePoint
  BioReplicate
  Experiment
  Mixture
  TechRepMixture
  Batch
  Channel
  Fraction
  Species
  Disease
  Tissue
  Enzyme
  Notes

First 5 rows:
        Dataset                            Run               Cohort Subject  \
0  MSV000085507  F20200310sunr_COVID_b1_1.mzML  Non-severe_COVID-19     XG1   
1  MSV000085507  F20200310sunr_COVID_b1_1.mzML  Non-severe_COVID-19     XG2   
2  MSV000085507  F20200310sunr_COVID_b1_1.mzML  Non-severe_COVID-19     XG3   
3  MSV000085507  F20200310sunr_COVID_b1_1.mzML      Severe_COVID-19    XG26   
4  MSV000085507  F20200310sunr_COVID_b1_1.mzML      Severe_COVID-19    XG27   

             Condition  TimePoint BioReplicate  Experiment Mixture  \
0  Non-severe_COVID-19        NaN          XG1           1      b1   
1  Non-severe_COVID-19        NaN          XG2           1      b1   
2  Non-severe_

## Step 4: Patient and Condition Distribution

We count how many patients and MS runs belong to each condition. Note that `Empty` and `Norm` rows are not real patients — they represent empty channels and normalization references used in TMT isobaric labeling workflows.

In [18]:
# Row counts per condition (each row = one channel/run combination)
print("Rows per condition:")
print(df['Condition'].value_counts())

# Unique patients per condition
print("\nUnique subjects per condition:")
print(df.groupby('Condition')['Subject'].nunique())

print(f"\nTotal unique subjects (including Empty/Norm placeholders): {df['Subject'].nunique()}")
print(f"Total unique MS runs: {df['Run'].nunique()}")

# Verify each subject appears in exactly one condition
subject_conditions = df.groupby('Subject')['Condition'].nunique()
multi = subject_conditions[subject_conditions > 1]
if multi.empty:
    print("\nAll subjects belong to exactly one condition. ✓")
else:
    print("\nSubjects appearing in multiple conditions (unexpected):")
    print(multi)

Rows per condition:
Condition
Non-severe_COVID-19         1040
Symptomatic_non-COVID-19    1040
Healthy                      960
Empty                        920
Severe_COVID-19              840
Norm                         320
Name: count, dtype: int64

Unique subjects per condition:
Condition
Empty                        1
Healthy                     22
Non-severe_COVID-19         25
Norm                         1
Severe_COVID-19             18
Symptomatic_non-COVID-19    25
Name: Subject, dtype: int64

Total unique subjects (including Empty/Norm placeholders): 92
Total unique MS runs: 320

All subjects belong to exactly one condition. ✓


## Step 5: Build a Clean Patient-Condition Table

We filter out the `Empty` and `Norm` placeholder rows to get a clean mapping of real patients to their disease condition. This label table will be used downstream when training classifiers.

In [ ]:
real_patients = df[~df['Condition'].isin(['Empty', 'Norm'])]

print("Patient counts per condition (after removing Empty/Norm):")
print(real_patients.groupby('Condition')['Subject'].nunique())
print(f"\nTotal real patients: {real_patients['Subject'].nunique()}")

# One row per patient with their condition label
patient_labels = (
    real_patients
    .drop_duplicates('Subject')[['Subject', 'Condition']]
    .sort_values('Condition')
    .reset_index(drop=True)
)
print(f"\nPatient-condition table shape: {patient_labels.shape}")
print(patient_labels)

## Step 6: Check data sparsity

In [24]:
quant_df = pd.read_csv("quant_data.tsv", sep="\t")
# % of features with at least one missing value
percent_missing_features = (quant_df.isna().any(axis=0).mean()) * 100

# % of features present in <10% of samples
presence = (quant_df.notna().sum(axis=0) / quant_df.shape[0])
low_presence = (presence < 0.1).mean() * 100
print(f"Percentage of features with at least one missing value: {percent_missing_features:.1f}%")
print(f"Percentage of features present in <10% of samples: {low_presence:.1f}%")

Percentage of features with at least one missing value: 68.0%
Percentage of features present in <10% of samples: 4.0%
